In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv("../data/cleaned/reviews_cleaned.csv")
print(df.shape)
df.head()
# Task 2 - Sentiment Analysis
sentiment_pipeline = pipeline("sentiment-analysis", 
                            model="distilbert-base-uncased-finetuned-sst-2-english")

def get_sentiment(text):
    if not isinstance(text, str) or len(text.strip()) < 5:
        return "NEUTRAL", 0.0
    result = sentiment_pipeline(text[:512])[0]
    return result['label'], round(result['score'], 4)

print("Running sentiment analysis...")
results = df['review'].apply(get_sentiment)
df['sentiment_label'] = [x[0] for x in results]
df['sentiment_score'] = [x[1] for x in results]

df.to_csv("../data/cleaned/reviews_with_sentiment.csv", index=False)
print(df['sentiment_label'].value_counts())
# Thematic Analysis
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=100)
tfidf = vectorizer.fit_transform(df['review'].dropna())

words = vectorizer.get_feature_names_out()
scores = tfidf.sum(axis=0).A1
top_keywords = sorted(zip(words, scores), key=lambda x: x[1], reverse=True)[:20]

print("Top Keywords:")
for word, score in top_keywords[:15]:
    print(f"{word}: {score:.3f}")

plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='bank', hue='sentiment_label')
plt.title('Sentiment Distribution by Bank')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Save for report
plt.savefig('../sentiment_by_bank.png')